In [27]:
import os
from dotenv import load_dotenv

# Load the variables from .env
load_dotenv()

# Quick check to make sure it worked
if "GOOGLE_API_KEY" not in os.environ:
    print("Warning: GOOGLE_API_KEY not found. Check your .env file!")
else:
    print("API Key loaded successfully!")

API Key loaded successfully!


In [30]:
from pathlib import Path
from langchain_community.document_loaders import CSVLoader, PDFPlumberLoader

def load_documents(csv_path: str, pdf_path: str):
    """Loads CSV and PDF files safely, catching errors if they occur."""
    documents = []
    
    # 1. Safely load the CSV
    if Path(csv_path).exists():
        print(f"Loading CSV from '{csv_path}'...")
        try:
            csv_loader = CSVLoader(file_path=csv_path)
            documents.extend(csv_loader.load())
            print("CSV loaded successfully.")
        except Exception as e:
            print(f"Error loading CSV: {e}")
    else:
        print(f"Warning: CSV file not found at '{csv_path}'")

    # 2. Safely load the PDF (Using PDFPlumber for better Windows compatibility and table parsing)
    if Path(pdf_path).exists():
        print(f"Loading PDF from '{pdf_path}'...")
        try:
            # PDFPlumber is fantastic for preserving tabular layouts
            pdf_loader = PDFPlumberLoader(file_path=pdf_path)
            documents.extend(pdf_loader.load())
            print("PDF loaded successfully.")
        except Exception as e:
            print(f"Error loading PDF: {e}")
    else:
        print(f"Warning: PDF file not found at '{pdf_path}'")
        
    return documents

# --- TEST IT ---
csv_filename = "docs/MY_CSV.csv" 
pdf_filename = "docs/knowledgeBase.pdf"

all_docs = load_documents(csv_filename, pdf_filename)

if all_docs:
    print(f"\nSuccess! Total document elements extracted: {len(all_docs)}")
else:
    print("\nNo documents were loaded. Please check your file paths.")

Loading CSV from 'docs/MY_CSV.csv'...
CSV loaded successfully.
Loading PDF from 'docs/knowledgeBase.pdf'...
PDF loaded successfully.

Success! Total document elements extracted: 53


In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_documents(documents):
    """Splits loaded documents into smaller, overlapping chunks."""
    if not documents:
        print("Error: No documents provided to chunk.")
        return []
        
    print(f"\nChunking {len(documents)} document elements...")
    
    try:
        # chunk_size: max characters per chunk
        # chunk_overlap: characters to overlap between chunks to keep context
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000, 
            chunk_overlap=200,
            # These separators tell the chunker to try splitting by paragraphs first, 
            # then lines, then sentences, keeping table rows together as much as possible.
            separators=["\n\n", "\n", r"(?<=\. )", " ", ""]
        )
        
        chunks = text_splitter.split_documents(documents)
        print(f"Successfully split into {len(chunks)} chunks.")
        return chunks
    except Exception as e:
        print(f"Critical error during chunking: {e}")
        return []

# --- TEST IT ---
# Add this right below your previous test code
my_chunks = chunk_documents(all_docs)

if my_chunks:
    # Print a tiny preview of the very first chunk to verify it's readable text
    print("\n--- Preview of Chunk 1 ---")
    print(repr(my_chunks[0].page_content[:150]) + "...")
    print("--------------------------")


Chunking 53 document elements...
Successfully split into 63 chunks.

--- Preview of Chunk 1 ---
'Industry: Accounting/Finance'...
--------------------------


In [32]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os # Make sure os is imported to grab the new key
def create_vector_db(chunks):
    """Converts text chunks into embeddings locally and stores them in a Chroma database."""
    print("\nInitializing Local HuggingFace Embeddings...")
    try:
        # This downloads a small (~90MB) model to your machine the first time you run it, 
        # then runs instantly on your CPU from then on. Zero rate limits.
        embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        
        print("Creating Chroma vector database locally...")
        vector_db = Chroma.from_documents(documents=chunks, embedding=embeddings)
        print("Vector database created successfully.")
        return vector_db
    except Exception as e:
        print(f"Critical error creating vector database: {e}")
        return None

# --- TEST IT ---
# Add this below your chunking test
my_vector_db = create_vector_db(my_chunks)

if my_vector_db:
    # Let's do a quick test search to prove the database actually works
    # Using a word we know exists based on your chunk preview
    test_query = "finance" 
    print(f"\nTesting database with query: '{test_query}'")
    
    # We ask the DB to find the top 2 chunks most mathematically similar to our query
    results = my_vector_db.similarity_search(test_query, k=2)
    print(f"Success! Found {len(results)} matching chunks.")


Initializing Local HuggingFace Embeddings...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1217.33it/s]


Creating Chroma vector database locally...
Vector database created successfully.

Testing database with query: 'finance'
Success! Found 2 matching chunks.


In [35]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

def setup_rag_chain(vector_db):
    """Connects the local vector database to Gemini 1.5 Flash."""
    print("\nInitializing Google Gemini LLM...")
    
    # Using Gemini 1.5 Flash - fast, smart, and handles tables beautifully
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.1-flash-lite",
        temperature=0.1
    )

    # Tell the database to return the 4 best matching chunks for any question
    retriever = vector_db.as_retriever(search_kwargs={"k": 15})

    # The strict instruction prompt
    system_prompt = (
        "You are a professional assistant analyzing documents. "
        "Use the following pieces of retrieved context to answer the user's question. "
        "If you cannot find the answer in the context, clearly state that you don't know based on the provided documents. "
        "Do not make up information. "
        "\n\n"
        "Context:\n{context}"
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])

    # Chain it together
    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)
    
    return rag_chain

# --- FINAL TEST ---
# Add this at the very bottom of your script
my_bot = setup_rag_chain(my_vector_db)

if my_bot:
    print("\n" + "="*50)
    print("🤖 RAG BOT IS ONLINE! Type 'quit' to exit.")
    print("="*50)
    
    while True:
        user_question = input("\nAsk a question about your files: ")
        if user_question.lower() in ['quit', 'exit']:
            print("Shutting down bot. Goodbye!")
            break
            
        print("Thinking...")
        try:
            # The bot searches the DB and generates an answer
            response = my_bot.invoke({"input": user_question})
            print(f"\nAnswer:\n{response['answer']}")
        except Exception as e:
            print(f"\nError generating answer: {e}")


Initializing Google Gemini LLM...

🤖 RAG BOT IS ONLINE! Type 'quit' to exit.


Thinking...

Answer:
I do not know the total number of customers based on the provided documents. The documents only list various industries and do not contain any numerical data regarding customer counts.
Thinking...

Answer:
Based on the provided documents, there is no information regarding the number of customers. The documents only list various industries.
Thinking...

Answer:
Based on the provided documents, Jason Maldonado is associated with booking ID **BK1100**. His contact email is georgemiller@example.com, he booked a Twin room (room number 573) for the period of 2026-06-11 to 2026-06-23, and his booking status is "Pending," "Under Maintenance," and "Refunded."
Shutting down bot. Goodbye!
